In [1]:
import re
import pandas as pd
from tqdm import tqdm

import os
tqdm.pandas()

In [7]:
df = pd.read_parquet("../data/datasets/MPV_612_2013_textos_emendas.parquet")
df

,num_emenda,materia,nome_arquivo,texto
0,100,MPV_612_2013,EMENDA_100_-_MPV_612_2013.txt,Subsecretaria de Apoio às Comissões Mistas\n\n...
1,101,MPV_612_2013,EMENDA_101_-_MPV_612_2013.txt,Mistas\n\nissões À\n\na2\n\nGigliola Ansiliero...
2,102,MPV_612_2013,EMENDA_102_-_MPV_612_2013.txt,Subsecretaria de Apoio às Comissões Mistas\nRe...
3,103,MPV_612_2013,EMENDA_103_-_MPV_612_2013.txt,SE\n\nMPV 612\n\nCONGRESSO NACIONAL 00103\n\nA...
4,104,MPV_612_2013,EMENDA_104_-_MPV_612_2013.txt,"Mista,\n\nàs 1) /\n\norais, Mat. 258286\n\n-)\..."
...,...,...,...,...
215,96,MPV_612_2013,EMENDA_96_-_MPV_612_2013.txt,; Congresso Nacional MEPV 612\n\n00096\nAPRESE...
216,97,MPV_612_2013,EMENDA_97_-_MPV_612_2013.txt,s\n\n€ Ápoio às Comissões Miistas\n\nDÃ\ndu G\...
217,98,MPV_612_2013,EMENDA_98_-_MPV_612_2013.txt,Subsecretaria de Apois às Comissões Mustas\n\n...
218,99,MPV_612_2013,EMENDA_99_-_MPV_612_2013.txt,Subsecretaria de Apoio às Comissões i1stas\n\n...


In [8]:
TXT_DIR = "../data/txts/MPV_612_2013/txts_from_label_studio_annotations"

textos_preprocessados = {}
textos_preprocessados_sem_justificativa = {}

for arquivo in os.listdir(TXT_DIR):

    if not arquivo.endswith(".txt"):
        continue

    caminho = os.path.join(TXT_DIR, arquivo)

    with open(caminho, "r", encoding="utf-8") as f:
        texto_original = f.read()

    # ------------------------
    # Texto preprocessado normal
    # ------------------------
    texto = texto_original

    texto = re.sub(
        r"===\s*PÁGINA\s+\d+\s*===",
        "",
        texto,
        flags=re.IGNORECASE
    )

    texto = re.sub(
        r"\[EMENDA\]",
        "",
        texto,
        flags=re.IGNORECASE
    )

    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = texto.strip()

    textos_preprocessados[arquivo] = texto

    # ------------------------
    # Texto sem justificativa
    # ------------------------
    texto = texto_original

    paginas = re.split(
        r"(===\s*PÁGINA\s+\d+\s*===)",
        texto,
        flags=re.IGNORECASE
    )

    paginas_limpas = []

    for i in range(1, len(paginas), 2):

        pagina = paginas[i + 1]

        pagina = re.sub(
            r"\[JUSTIFICATIVA\][\s\S]*",
            "",
            pagina,
            flags=re.IGNORECASE
        )

        pagina = re.sub(
            r"\[(EMENDA|JUSTIFICATIVA)\]",
            "",
            pagina,
            flags=re.IGNORECASE
        )

        pagina = pagina.strip()

        if pagina:
            paginas_limpas.append(pagina)

    texto_sem_justificativa = "\n\n".join(paginas_limpas)

    texto_sem_justificativa = re.sub(
        r"\n{3,}",
        "\n\n",
        texto_sem_justificativa
    )

    texto_sem_justificativa = texto_sem_justificativa.strip()

    textos_preprocessados_sem_justificativa[arquivo] = (
        texto_sem_justificativa
    )

print(f"Arquivos carregados: {len(textos_preprocessados)}")

df["texto_preprocessado"] = df["nome_arquivo"].map(
    textos_preprocessados
)

df["texto_preprocessado_sem_justificativa"] = (
    df["nome_arquivo"].map(
        textos_preprocessados_sem_justificativa
    )
)

print(
    "Com texto preprocessado:",
    df["texto_preprocessado"].notna().sum()
)

print(
    "Com texto sem justificativa:",
    df["texto_preprocessado_sem_justificativa"].notna().sum()
)

df[
    [
        "nome_arquivo",
        "texto_preprocessado",
        "texto_preprocessado_sem_justificativa",
    ]
].head()

Arquivos carregados: 220
Com texto preprocessado: 220
Com texto sem justificativa: 220


,nome_arquivo,texto_preprocessado,texto_preprocessado_sem_justificativa
0,EMENDA_100_-_MPV_612_2013.txt,"O Art, 23 da Medida Provisória n.º 612, de 04 ...","O Art, 23 da Medida Provisória n.º 612, de 04 ..."
1,EMENDA_101_-_MPV_612_2013.txt,"O Art. 23 da Medida Provisória n.º 612, de 04 ...","O Art. 23 da Medida Provisória n.º 612, de 04 ..."
2,EMENDA_102_-_MPV_612_2013.txt,"Modifica as alíneas “a” e “b” do incido 1l, do...","Modifica as alíneas “a” e “b” do incido 1l, do..."
3,EMENDA_103_-_MPV_612_2013.txt,"O artigo 25 da MPV 612, de 2013, que altera o ...","O artigo 25 da MPV 612, de 2013, que altera o ..."
4,EMENDA_104_-_MPV_612_2013.txt,"Acrescente-se o inciso Ill, no Art. 26 da Medi...","Acrescente-se o inciso Ill, no Art. 26 da Medi..."


In [9]:
df[
    [
        "nome_arquivo",
        "texto_preprocessado",
        "texto_preprocessado_sem_justificativa",
    ]
].head().values

array([['EMENDA_100_-_MPV_612_2013.txt',
        'O Art, 23 da Medida Provisória n.º 612, de 04 de Abril de 2013, passa a vigorar com a seguinte redação:\n\nArt. 283. A Lei nº 12.715, de 17 de setembro de 2012, passa a vigorar com as seguintes alterações:\n\n8) ficam limitadas a três por cento do imposto sobre a renda devido com relação aào programa de que trata\n| O art, 1º, e a um por cento do imposto sobre a renda devido com relação ao programa de que trata o art. 3º;\n\nd) ficam limitadas a três por cento do imposto sobre a renda devido em cada período de apuração\ntrimestral ou anual com relação ao programa de que trata o art. 1º, e a um por cento do imposto sobre a renda\ndevido em cada período de apuração trimestral! ou anual com relação ao programa de que trata o art. 3º,\nobservado em ambas as hipóteses o disposto no $ 4º do art. 3º da Lei nº 9.249, de 26 de dezembro de 1995.\n\n....................................................................................\n\n[JUSTIFICAT

In [10]:
df[df.texto_preprocessado == df.texto_preprocessado_sem_justificativa]

,num_emenda,materia,nome_arquivo,texto,texto_preprocessado,texto_preprocessado_sem_justificativa


In [11]:
df.to_parquet("../data/datasets/MPV_612_2013_preprocessado.parquet", index=False)